# detach-stop-gradient-trick — worked example 2: Detach changes the backward graph but not the forward value

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-stop-gradient-trick`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A common worry is that `.detach()` somehow alters the numbers. It does not: `G(z).detach()` is numerically identical to `G(z)`. The only difference is that the detached tensor has `requires_grad=False` and `grad_fn=None`, so it is invisible to `backward()`. This is why swapping `G(z)` for `G(z).detach()` in the D-step never changes the loss you observe.

## Worked solution

**Goal:** show that detaching leaves the forward pass byte-for-byte identical while removing it from autograd.

1. **Single generator.** Build one `nn.Linear` as G and draw a fixed input `z` after re-seeding.
2. **Compute both versions of the fake.** `with_graph = G(z)` keeps the graph (`requires_grad=True`, has a `grad_fn`). `detached = G(z).detach()` produces the identical values but drops the graph.
3. **Compare values.** `torch.allclose(with_graph, detached)` is `True` — detach copies the data, only the metadata differs.
4. **Compare metadata.** `with_graph.requires_grad` is `True` and `with_graph.grad_fn` is not `None`; `detached.requires_grad` is `False` and `detached.grad_fn` is `None`.
5. **Why it matters for GANs.** Because the forward value is unchanged, the discriminator sees exactly the same fake image either way — you only gate whether the generator's parameters get a gradient. So you can drop in `.detach()` with zero effect on D's loss magnitude.

We print the allclose result plus the two `requires_grad` flags to make the distinction explicit.

In [ ]:
import torch.nn as nn

t.manual_seed(0)
G = nn.Linear(5, 3)
z = t.randn(6, 5)

with_graph = G(z)
detached = G(z).detach()

same_values = t.allclose(with_graph, detached)
print('forward values equal:', same_values)
print('with_graph.requires_grad:', with_graph.requires_grad)
print('detached.requires_grad :', detached.requires_grad)
print('detached.grad_fn is None:', detached.grad_fn is None)